# Gemini 를 OpenAI 라이브러리로 부르기

앞 노트북(`1_1_openai`)과 **거의 같은 코드**입니다. 그런데 답하는 건 구글 Gemini 입니다.

어떻게 가능할까요? 많은 LLM 회사가 **OpenAI 와 같은 모양의 API**를 함께 제공하기 때문입니다.
이것을 'OpenAI 호환(compatible) 엔드포인트' 라고 부릅니다.

## 1_1 과 무엇이 다른가

| | `1_1_openai` | 이 노트북 |
|---|---|---|
| 라이브러리 | `openai` | `openai` **(같다)** |
| 호출 코드 | `chat.completions.create(...)` | **(같다)** |
| `api_key` | OpenAI 키 | 구글 키 |
| `base_url` | 기본값 (생략) | **구글 주소** |
| `model` | `gpt-...` | `gemini-2.5-flash` |

> **바뀐 것은 딱 세 줄입니다.** 나머지 코드는 그대로입니다.
> 이것이 이 노트북의 전부이자 핵심입니다.

## Step 1. 가져오는 것은 그대로

`google` 패키지가 아니라 **`openai` 패키지를 그대로** 씁니다.
구글 모델을 쓰는데 OpenAI 라이브러리를 가져오는 것이 어색해 보이지만, 이게 요점입니다.

> 구글 전용 라이브러리(`google-genai`)도 있습니다. 기능을 모두 쓰려면 그쪽이 낫습니다.
> 여기서는 **여러 회사 모델을 같은 코드로 다루는 방법**을 배우는 것이 목적입니다.

In [1]:
import os
#import requests
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. `.env` 읽기

`1_1` 과 똑같습니다. `True` 가 나오면 `.env` 파일을 제대로 찾았다는 뜻입니다.

In [2]:
load_dotenv(override=True)

True

## Step 3. 구글 API 키 확인

`OPENAI_API_KEY` 가 아니라 **`GOOGLE_API_KEY`** 를 읽습니다.
키는 [Google AI Studio](https://aistudio.google.com/apikey) 에서 발급받아 `.env` 에 적어 둡니다.

```
GOOGLE_API_KEY=AI...
```

> 메시지에 `(and this is optional)` 이 붙은 이유는
> 이 교육 과정에서 Gemini 가 필수가 아니기 때문입니다. 없으면 이 노트북만 건너뛰면 됩니다.

In [3]:
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

Google API Key exists and begins AQ


## Step 4. 주소(base_url) — 이 노트북의 핵심

```
https://generativelanguage.googleapis.com/v1beta/openai/
```

주소 끝에 **`/openai/`** 가 붙어 있는 것을 보세요.
"OpenAI 와 같은 모양으로 받아 주는 입구"라는 뜻입니다.

| 회사 | base_url |
|---|---|
| OpenAI | (생략 — 라이브러리 기본값) |
| Google | `https://generativelanguage.googleapis.com/v1beta/openai/` |
| Ollama (내 PC) | `http://localhost:11434/v1` |

주소도 `.env` 에 넣어 두었습니다. 코드에 박아 두면 바뀔 때마다 코드를 고쳐야 하기 때문입니다.

In [4]:
gemini_url = os.getenv('GEMINI_URL')
print(gemini_url)

https://generativelanguage.googleapis.com/v1beta/openai/


## Step 5. 클라이언트 만들기 — 두 줄이 전부

```python
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
```

`1_1` 에서는 `OpenAI()` 로 비워 뒀습니다. 환경변수에서 OpenAI 키를 자동으로 찾기 때문입니다.
여기서는 **다른 회사**를 쓰므로 키와 주소를 직접 알려 줘야 합니다.

변수 이름을 `gemini` 로 지었지만 **타입은 여전히 `OpenAI`** 입니다.
껍데기는 OpenAI 라이브러리이고, 안에서 말을 거는 상대만 구글로 바뀐 것입니다.

In [5]:
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

## Step 6. `system` 프롬프트

`1_1_openai` 와 **글자 하나까지 똑같은** 프롬프트를 씁니다.
같은 질문을 다른 모델에 던져 **답이 어떻게 다른지 비교**하려는 것입니다.

In [9]:
system_prompt = """
당신은 IT 보안 전문가입니다. 
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
"""

## Step 7. `user` 프롬프트

이것도 `1_1` 과 같습니다. 비교가 목적이므로 조건을 똑같이 맞춥니다.

In [10]:
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?

"""

## Step 8. 호출 — 모델 이름만 바뀐다

```python
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=message)
```

`1_1` 의 호출과 비교해 보세요.

```python
# 1_1
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
# 여기
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=message)
```

**클라이언트 변수와 모델 이름만 다릅니다.** 함수도, 인자 이름도, `messages` 구조도 같습니다.

| 모델 이름 | 성격 |
|---|---|
| `gemini-2.5-flash` | 빠르고 싸다. 일상적인 작업용 |
| `gemini-2.5-pro` | 느리고 비싸다. 어려운 추론용 |

In [16]:
message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

response = gemini.chat.completions.create(model="gemini-3.8-flash", messages=message)


## Step 9. 응답 구조도 같은 모양

```python
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))
```

구글이 답했는데도 **`choices[0].message.content`, `usage`, `finish_reason`** 이 그대로 있습니다.
OpenAI 호환이라는 말은 **응답 모양까지 같다**는 뜻입니다.

덕분에 응답을 처리하는 코드를 모델마다 새로 짜지 않아도 됩니다.

> `id` 값은 OpenAI 의 `chatcmpl-...` 과 형식이 다릅니다.
> 호환이라 해도 내부 값까지 같지는 않습니다.

In [17]:
import json

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "TnjLasmOFKWB0-kPh7ntoQE",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "보안 책임자(CISO/보안 리드)로서 회사의 자산과 비즈니스 연속성을 지키기 위해 많은 고민이 있으실 것으로 생각됩니다. \n\n현재 글로벌 및 국내 IT 업계에서 **가장 시급하고 파괴력이 큰 핵심 보안 이슈 4가지**와 이에 대한 **전문가 관점의 대응 전략**을 정리해 드립니다.\n\n---\n\n### 1. 생성형 AI 악용 및 '섀도우 AI(Shadow AI)' 리스크\n최근 가장 급격하게 부상한 위협입니다. 공격자뿐만 아니라 내부 임직원에 의해서도 보안 사고가 발생하고 있습니다.\n\n*   **주요 위협:**\n    *   **공격의 고도화:** LLM을 이용해 문법적 오류가 전혀 없는 정교한 스피어 피싱(Spear Phishing), 악성코드 제작, 음성·영상 딥페이크를 통한 CEO 사칭(BEC 공격).\n    *   **데이터 유출:** 사내 개발자나 기획자가 ChatGPT, Copilot 등에 회사의 핵심 소스코드나 고객 개인정보를 무단 입력하여 외부 모델 학습에 노출되는 문제.\n*   **대응 방안:**\n    *   **AI 거버넌스 및 정책 수립:** 생성형 AI 사용 가이드라인을 즉시 수립하고, 승인되지 않은 AI 툴 접근 차단(CASB 활용).\n    *   **데이터 유출 방지(DLP):** 엔드포인트 및 네트워크 단에서 프롬프트 입력 창으로 민감 정보(코드, API 키 등)가 전송되지 않도록 필터링 규칙 적용.\n    *   **임직원 훈련:** 텍스트뿐만 아니라 음성/영상 딥페이크 공격에 대비한 다채널 검증 프로세스(중요 송금 시 대면/유선 교차 확인 등) 도입.\n\n### 2. 소프트웨어 공급망(Software Su

## Step 10. 답변 꺼내기

```python
print(response.choices[0].message.content)
```

`1_1` 과 똑같은 한 줄입니다.

Gemini 의 답을 `1_1_openai` 의 답과 비교해 보세요. 같은 질문인데 **구성과 깊이가 다릅니다.**
어느 쪽이 낫다기보다 모델마다 성격이 다르다고 보는 편이 맞습니다.
그래서 실무에서는 **용도별로 모델을 골라 씁니다.**

In [18]:
print(response.choices[0].message.content)

보안 책임자(CISO/보안 리드)로서 회사의 자산과 비즈니스 연속성을 지키기 위해 많은 고민이 있으실 것으로 생각됩니다. 

현재 글로벌 및 국내 IT 업계에서 **가장 시급하고 파괴력이 큰 핵심 보안 이슈 4가지**와 이에 대한 **전문가 관점의 대응 전략**을 정리해 드립니다.

---

### 1. 생성형 AI 악용 및 '섀도우 AI(Shadow AI)' 리스크
최근 가장 급격하게 부상한 위협입니다. 공격자뿐만 아니라 내부 임직원에 의해서도 보안 사고가 발생하고 있습니다.

*   **주요 위협:**
    *   **공격의 고도화:** LLM을 이용해 문법적 오류가 전혀 없는 정교한 스피어 피싱(Spear Phishing), 악성코드 제작, 음성·영상 딥페이크를 통한 CEO 사칭(BEC 공격).
    *   **데이터 유출:** 사내 개발자나 기획자가 ChatGPT, Copilot 등에 회사의 핵심 소스코드나 고객 개인정보를 무단 입력하여 외부 모델 학습에 노출되는 문제.
*   **대응 방안:**
    *   **AI 거버넌스 및 정책 수립:** 생성형 AI 사용 가이드라인을 즉시 수립하고, 승인되지 않은 AI 툴 접근 차단(CASB 활용).
    *   **데이터 유출 방지(DLP):** 엔드포인트 및 네트워크 단에서 프롬프트 입력 창으로 민감 정보(코드, API 키 등)가 전송되지 않도록 필터링 규칙 적용.
    *   **임직원 훈련:** 텍스트뿐만 아니라 음성/영상 딥페이크 공격에 대비한 다채널 검증 프로세스(중요 송금 시 대면/유선 교차 확인 등) 도입.

### 2. 소프트웨어 공급망(Software Supply Chain) 공격의 일상화
IT 기업은 소프트웨어를 '만드는 주체'이자 '사용하는 주체'이기 때문에 공급망 공격의 최전선에 있습니다. 최근 오픈소스 유틸리티 `xz-utils` 백도어 사태처럼 오픈소스 생태계를 직접 겨냥하는 위협이 현실화되었습니다.

*   **주요 위협:**
    *   CI/CD 파이프라인 침투, 오픈

---

## 정리

```python
OpenAI(api_key=..., base_url=...)   # 이 두 인자가 전부
```

| | 내용 |
|---|---|
| **라이브러리는 하나로** | `openai` 하나로 여러 회사 모델을 부를 수 있다 |
| **바뀌는 건 세 가지** | `api_key` · `base_url` · `model` |
| **응답 모양도 같다** | 처리 코드를 다시 짤 필요가 없다 |

### 직접 해 보기

1. `model` 을 `gemini-2.5-pro` 로 바꿔 보세요. 답과 `usage` 가 어떻게 달라지나요?
2. `base_url` 을 일부러 틀리게 적어 보세요. 어떤 오류가 나나요?
3. `1_1_openai` 의 답과 나란히 놓고 비교해 보세요.

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_3_ollama` | 같은 방법으로 **내 PC 안에서 도는** 모델 부르기 |
| `1_4_llms` | 네 가지 모델에 같은 질문을 던져 한 번에 비교 |